# DIRE trên Kaggle — preflight dữ liệu chỉ đọc trước khi chốt compute

Notebook này bắt đầu bằng **preflight chỉ đọc**, trước mọi smoke/train cell. Preflight xác minh manifest, SHA-256/kích thước/member counts của 8 archive CelebA-HQ, dung lượng trống và khả năng ghi vào `/kaggle/working`; `nvidia-smi` chỉ được truy vấn để đọc thông tin GPU.

**Preflight không** giải nén dữ liệu, ghi vào `/kaggle/input`, chạy workload GPU, kiểm tra quota tài khoản, xác minh provenance/pixel identity, hoặc chứng minh artifact đã được lưu bền vững qua phiên. Báo cáo phải được xuất/lưu bằng cơ chế Kaggle sau khi kiểm tra.

**Trước khi chạy:**
1. Chọn *Settings → Accelerator → Off* (preflight không cần GPU).
2. Gắn Kaggle Dataset versioned/immutable chứa `kaggle-map-manifest.json` và các archive dưới `dire/`.
3. Cài/lấy source repo bằng cell 2, rồi chạy preflight cell ngay sau đó. Không chạy smoke/train cell nếu preflight báo fail.

Số liệu của archive chỉ được coi là xác minh sau khi preflight chạy trên dataset đã mount; không suy ra training budget từ preflight này.


In [ ]:
# 1. Kiểm tra Python, đĩa và mounted inputs — không khởi tạo CUDA/GPU
import os, sys, subprocess, glob, json

print('python:', sys.version.split()[0])
print(subprocess.run(['df', '-h', '/kaggle/working'], capture_output=True, text=True).stdout)
print('Mounted inputs:')
for path in sorted(glob.glob('/kaggle/input/*'))[:30]:
    print(' -', path)


In [ ]:
# 2. Cài phụ thuộc + lấy code repo
# Repo riêng tư → upload file zip vào Kaggle Dataset, hoặc pip từ git nếu bạn để public.
import subprocess, sys, glob, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'webdataset', 'pillow', 'pyyaml', 'boto3'], check=True)

REPO_ZIP = glob.glob('/kaggle/input/*/dire-modal*.zip') + glob.glob('/kaggle/input/*/*/dire-modal*.zip')
if REPO_ZIP:
    subprocess.run(['unzip', '-q', '-o', REPO_ZIP[0], '-d', '/kaggle/working/repo'], check=True)
    os.chdir('/kaggle/working/repo/dire-modal' if os.path.isdir('/kaggle/working/repo/dire-modal') else '/kaggle/working/repo')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
    print('cài từ zip:', os.getcwd())
else:
    # Nếu repo public (không có secret nào trong đó):
    GIT = 'https://github.com/jackboom2510/dire-modal'
    print(f'không thấy zip trong /kaggle/input — thử pip từ git: {GIT}')
    try:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'git+{GIT}'], check=True)
    except subprocess.CalledProcessError as exc:
        print('⚠ Không cài được (repo private?). Hãy upload zip vào Kaggle Dataset:', exc)

sys.path.insert(0, os.getcwd())

In [ ]:
# Read-only research data preflight — run before any optional smoke/train cell
from pathlib import Path
import glob
from dire_modal.kaggle_preflight import preflight_dataset

manifest_candidates = sorted(Path(p) for p in glob.glob(
    '/kaggle/input/**/kaggle-map-manifest.json', recursive=True
))
if len(manifest_candidates) != 1:
    raise RuntimeError(
        f'Expected exactly one kaggle-map-manifest.json in mounted Kaggle inputs; '
        f'found {len(manifest_candidates)}: {manifest_candidates}'
    )
MANIFEST_PATH = manifest_candidates[0]
DATA_ROOT = MANIFEST_PATH.parent
PREFLIGHT_WORKDIR = Path('/kaggle/working/dire-preflight')
PREFLIGHT_REPORT = PREFLIGHT_WORKDIR / 'preflight-report.json'

preflight_report = preflight_dataset(
    DATA_ROOT,
    MANIFEST_PATH,
    PREFLIGHT_WORKDIR,
    output_path=PREFLIGHT_REPORT,
    check_gpu=True,  # nvidia-smi query only; no CUDA context/model/workload
)
print(f"Preflight status: {preflight_report['status']}")
print(f"Report: {PREFLIGHT_REPORT}")
print(f"Writable workdir: {preflight_report.get('working_storage', {}).get('writable')}")
print(f"Free bytes: {preflight_report.get('working_storage', {}).get('free_bytes')}")
print(f"GPU info (query only): {preflight_report.get('gpu_info')}")
for error in preflight_report.get('errors', []):
    print('ERROR:', error)
for warning in preflight_report.get('warnings', []):
    print('NOTE:', warning)
if preflight_report['status'] != 'pass':
    raise RuntimeError('Preflight failed; do not continue to smoke/train cells.')


In [ ]:
# 3. Dataset đã được phát hiện và kiểm tra bởi preflight; không tải/ghi input
print('DATA_ROOT =', DATA_ROOT)
print('Manifest =', MANIFEST_PATH)
print('Preflight report =', PREFLIGHT_REPORT)
print('Archive total bytes =', sum(row.get('archive_bytes', 0) for row in preflight_report['archives']))


In [ ]:
# 4. (Tùy chọn) Pack 1 archive nhỏ → WebDataset shard (kiểm tra logic pack.py chạy được)
import tarfile, io, os, json
from pathlib import Path

OUT = Path('/kaggle/working/shards'); OUT.mkdir(exist_ok=True)

archives = sorted(Path(DATA_ROOT).rglob('*.tar.gz'))
if not archives:
    print('⚠ Không có archive nào — bỏ qua cell này (xem bước 5 của runbook)')
else:
    src = min(archives, key=lambda p: p.stat().st_size)  # nhỏ nhất để chạy nhanh
    print(f'pack {src.name} ({src.stat().st_size/1e6:.1f} MB)')
    import webdataset as wds
    out_dir = OUT / src.name.replace('.tar.gz', ''); out_dir.mkdir(parents=True, exist_ok=True)
    label = 0 if '/real/' in str(src) or src.name.startswith('real') else 1
    n = 0
    with wds.ShardWriter(str(out_dir / 'shard-%06d.tar'), maxcount=500) as sink, tarfile.open(src, 'r:gz') as tf:
        for m in tf:
            if not m.isfile():
                continue
            name = m.name.lower()
            if not name.endswith(('.png', '.jpg', '.jpeg')):
                continue
            sink.write({'__key__': f'{src.stem}/{n:08d}', name.rsplit('.', 1)[-1]: tf.extractfile(m).read(), 'cls': str(label).encode()})
            n += 1
    (out_dir / 'meta.json').write_text(json.dumps({'src': str(src), 'count': n, 'label': label}, indent=2))
    print(f'[ok] {n} ảnh → {out_dir}')

In [ ]:
# 5. (Tùy chọn) DataLoader + 1 bước train giả (kiểm tra đường đi dữ liệu end-to-end)
import torch, torch.nn as nn, torchvision.transforms as T
from PIL import Image
import webdataset as wds, glob

shards = sorted(glob.glob('/kaggle/working/shards/**/*.tar', recursive=True))
if not shards:
    print('⚠ Không có shard — chạy cell 4 trước (hoặc bỏ qua nếu chưa có dữ liệu)')
else:
    tf = T.Compose([T.Resize(256), T.CenterCrop(256), T.ToTensor()])

    def decode(sample):
        img = sample.get('png') or sample.get('jpg') or sample.get('jpeg')
        x = tf(Image.open(__import__('io').BytesIO(img)).convert('RGB'))
        y = int(sample['cls'].decode() if isinstance(sample['cls'], bytes) else sample['cls'])
        return x, y

    ds = wds.WebDataset(shards).decode('rgb').map(decode).batched(16)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = torchvision.models.resnet18(num_classes=2) if False else None
    import torchvision
    model = torchvision.models.resnet18(num_classes=2).to(dev)
    opt = torch.optim.SGD(model.parameters(), lr=1e-3)
    crit = nn.CrossEntropyLoss()

    steps = 0
    for x, y in ds:
        x, y = x.to(dev), y.to(dev)
        loss = crit(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
        steps += 1
        print(f'step {steps}: loss={loss.item():.4f} batch={tuple(x.shape)} device={dev}')
        if steps >= 3:
            break
    print(f'\n[ok] đường đi dữ liệu hoạt động — {steps} bước trên {dev}')
    print('→ Giờ có thể tin code trước khi tiêu credit Modal để train thật.')

## Sau preflight

- Chỉ tiếp tục smoke test khi preflight pass và người dùng chủ động chọn chạy cell tương ứng; các cell smoke hiện có tạo shard và thực hiện bước train giả, nên **không thuộc preflight**.
- Lưu/xuất report và mọi checkpoint/ledger cần resume riêng qua cơ chế Kaggle (ví dụ notebook output hoặc Dataset version); `/kaggle/working` không tự đảm bảo bền vững sau khi phiên bị xoá.
- Chưa được khóa seed, số lần chạy, epoch/batch/LR/patience hay training budget từ kết quả preflight. Không chạy benchmark/training במסגרת bước này.
